# **New York City Taxi Fare Prediction** #

# **I. Pendahuluan**

Dataset ini berisi data perjalanan taxi di New York City, dengan beberapa informasi penting seperti pickup_datetime, pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude, passenger_count, dan tentu saja fare_amount yang jadi target dari prediksi kita.

Setiap baris merepresentasikan satu perjalanan taxi, dan data ini bisa digunakan untuk mempelajari pola-pola perjalanan, pricing behavior, dan bahkan aspek geografis dari kota NYC. Proyek ini fokus untuk membangun model machine learning yang bisa memprediksi tarif taxi hanya berdasarkan informasi yang tersedia sebelum perjalanan terjadi.

Tujuan: Regresi

Memprediksi fare amount (jumlah tarif) dari suatu perjalanan taxi berdasarkan fitur-fitur seperti lokasi pickup dan dropoff, waktu, dan jumlah penumpang.

# **II. Paparan data**


## *II.a Library*

In [5]:
import os

# Pindah ke folder D:
os.chdir("D:/Kuliah/Taxi")

# Lihat isi folder untuk cek
print(os.listdir())


['GCP-Coupons-Instructions.rtf', 'sample_submission.csv', 'test.csv', 'train.csv']


In [20]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import radians, cos, sin, asin, sqrt
from pyspark.sql.functions import col, when, count
from pyspark.sql.functions import col, year, month, dayofmonth, hour, minute, second, dayofweek, to_timestamp

## *II.b Dataset*

In [8]:
spark = SparkSession.builder \
    .appName("TaxiProject") \
    .config("spark.local.dir", "D:/spark-temp") \
    .getOrCreate()


df = spark.read.csv("D:/Kuliah/Taxi/train.csv", header=True, inferSchema=True)
df.show(5)

+-------------------+-----------+-------------------+----------------+---------------+-----------------+----------------+---------------+
|                key|fare_amount|    pickup_datetime|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|passenger_count|
+-------------------+-----------+-------------------+----------------+---------------+-----------------+----------------+---------------+
|2009-06-15 17:26:21|        4.5|2009-06-16 00:26:21|      -73.844311|      40.721319|        -73.84161|       40.712278|              1|
|2010-01-05 16:52:16|       16.9|2010-01-05 23:52:16|      -74.016048|      40.711303|       -73.979268|       40.782004|              1|
|2011-08-18 00:35:00|        5.7|2011-08-18 07:35:00|      -73.982738|       40.76127|       -73.991242|       40.750562|              2|
|2012-04-21 04:30:42|        7.7|2012-04-21 11:30:42|       -73.98713|      40.733143|       -73.991567|       40.758092|              1|
|2010-03-09 07:51:00|        5.3|2

## *II.c Data Cleaning*

In [10]:
df = df.dropDuplicates()

In [11]:
# Check for missing values
missing_values = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
missing_values.show()

+---+-----------+---------------+----------------+---------------+-----------------+----------------+---------------+
|key|fare_amount|pickup_datetime|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|passenger_count|
+---+-----------+---------------+----------------+---------------+-----------------+----------------+---------------+
|  0|          0|              0|               0|              0|              376|             376|              0|
+---+-----------+---------------+----------------+---------------+-----------------+----------------+---------------+



In [14]:
# Remove rows with missing values
df = df.dropna()

# Filter out rows with negative fares or zero passengers
df = df.filter((df.fare_amount > 0) & (df.passenger_count > 0))

Convert to Distance

In [18]:
# Define a function to calculate Haversine distance
def haversine(lat1, lon1, lat2, lon2):
    return 2 * 6371 * asin(sqrt(sin((lat2 - lat1) / 2)**2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2)**2))

# Add a new column 'distance' using the Haversine formula
df = df.withColumn("distance", 
                               haversine(radians(df.pickup_latitude), radians(df.pickup_longitude),
                                         radians(df.dropoff_latitude), radians(df.dropoff_longitude)))

In [22]:
# Ubah kolom 'key' ke tipe timestamp
df = df.withColumn("key", to_timestamp("key"))

# Ekstrak bagian waktu
df = df.withColumn("year", year(col("key"))) \
       .withColumn("month", month(col("key"))) \
       .withColumn("day", dayofmonth(col("key"))) \
       .withColumn("hour", hour(col("key"))) \
       .withColumn("minute", minute(col("key"))) \
       .withColumn("second", second(col("key"))) \
       .withColumn("day_of_week", dayofweek(col("key")))  # Note: Minggu = 1, Senin = 2, dst


In [26]:
df = df.drop("key", "pickup_datetime")

In [32]:
df = df.drop("pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude")

In [34]:
df.show(5)

+-----------+---------------+------------------+----+-----+---+----+------+------+-----------+
|fare_amount|passenger_count|          distance|year|month|day|hour|minute|second|day_of_week|
+-----------+---------------+------------------+----+-----+---+----+------+------+-----------+
|        8.1|              1|1.5507439489530872|2011|    7| 26|  11|    56|    41|          3|
|        9.0|              2| 1.935718144226888|2014|    1|  4|  21|    11|     0|          7|
|      31.83|              1|  9.83427178131614|2013|    7| 19|  18|    55|     0|          6|
|       18.0|              1| 7.169858703189975|2012|   11| 28|   8|    30|     0|          4|
|       16.0|              1| 3.791956837345176|2012|   11| 25|   0|     9|     0|          1|
+-----------+---------------+------------------+----+-----+---+----+------+------+-----------+
only showing top 5 rows

